# Casos de Câncer de Mama — Análise Exploratória de Dados

**Projeto:** TCC — Análise de Dados de Câncer de Mama em Municípios de Minas Gerais  
**Metodologia:** KDD (Knowledge Discovery in Databases)  
**Etapa:** Análise Exploratória de Dados (EDA)  
**Escopo:** Incidência por Ano de Diagnóstico, Faixa Etária, Escolaridade e Raça/Cor (1985–2023) — municípios e regionais de saúde de MG

---
## 1. Imports

In [1]:
import json
import re
import unicodedata
import warnings
from pathlib import Path

import geobr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook_connected"

---
## 2. Configurações Gerais

Centraliza todos os parâmetros configuráveis: caminhos, filtros, paletas de cores e constantes de layout.  
Ajuste aqui qualquer parâmetro antes de executar o notebook.

In [2]:
# ── Filtros de escopo (None = sem filtro, analisa MG inteiro) ─────────────────
MUNICIPIO        = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE   = None

# ── Caminhos dos arquivos de entrada ─────────────────────────────────────────
CSV_CASOS_ANO_DIAGNOSTICO = '/home/anna/tcc/Tratamento/Casos/incidencia _ano_diagnostico.csv'
CSV_CASOS_ESCOLARIDADE    = '/home/anna/tcc/Tratamento/Casos/incidencia _escolaridade.csv'
CSV_CASOS_RACA_COR        = '/home/anna/tcc/Tratamento/Casos/incidencia _raca_cor.csv'
CSV_CASOS_FAIXA_ETARIA    = '/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv'

# ── Período de análise ────────────────────────────────────────────────────────
ANOS_RANGE_CASOS = (1985, 2023)
ANO_INI_MAPA, ANO_FIM_MAPA = ANOS_RANGE_CASOS
ANO_MALHA_MUNICIPIOS_MG    = 2020

# ── Categorias ───────────────────────────────────────────────────────────────
TOP_N_CATEGORIAS    = 25
FAIXAS_ETARIAS_ORDEM = ['0-9','10-19','20-29','30-39','40-49','50-59','60-69','70 ou mais']

# ── Layout dos gráficos ───────────────────────────────────────────────────────
COR_BARRAS         = "#6ca8e5"
COR_PONTOS         = "#d73f3f"
COR_TEXTO_BARRA    = "#1a1a1a"
LARGURA_GRAFICO    = 2300
ALTURA_GRAFICO     = 1700
X_MIN              = 0
FOLGA_ROTULOS      = 0.18     # margem à direita dos rótulos de pontos/barras

FONTE_TITULO       = 34
FONTE_EIXO_TITULO  = 28
FONTE_EIXO_TICK    = 18
FONTE_TEXTO_BARRA  = 20
FONTE_LEGENDA      = 22

# ── Paleta e faixas para os mapas coropléticos ───────────────────────────────
COR_SEM_CASOS     = "#6ca8e5"
PALETA_MAPA       = ["#9c2328", "#d20000", "#ff7f50", "#ee9c21", "#fec44f", "#ffffd4"]
FAIXAS_MUNICIPIOS = [100, 300, 600, 1000, 3000]
FAIXAS_REGIONAIS  = [500, 1000, 2500, 5000, 10000]

# ── Correção de grafia para categorias ───────────────────────────────────────
CORRECAO_GRAFIA = {
    'indigena':                  'Indígena',
    'parda':                     'Parda',
    'preta':                     'Preta',
    'branca':                    'Branca',
    'amarela':                   'Amarela',
    'ignorado':                  'Ignorado',
    'ignorada':                  'Ignorado',
    'sem informacao':            'Sem informação',
    'nenhuma':                   'Nenhuma',
    'fundamental incompleto':    'Fundamental incompleto',
    'fundamental completo':      'Fundamental completo',
    'medio incompleto':          'Médio incompleto',
    'medio completo':            'Médio completo',
    'nivel medio':               'Nível médio',
    'nivel medio incompleto':    'Nível médio incompleto',
    'nivel medio completo':      'Nível médio completo',
    'nivel superior incompleto': 'Nível superior incompleto',
    'nivel superior completo':   'Nível superior completo',
    'nao se aplica':             'Não se aplica',
    'nao informado':             'Não informado',
    'nao informada':             'Não informado',
}

---
## 3. Funções Auxiliares

Funções utilitárias reutilizadas ao longo de todo o notebook. Definidas uma única vez aqui para evitar repetição.

In [3]:
# ── Normalização de texto e colunas ──────────────────────────────────────────

def _norm_txt(txt: str) -> str:
    """Remove acentos e converte para minúsculas (para comparações)."""
    return (
        unicodedata.normalize('NFKD', str(txt))
        .encode('ascii', 'ignore').decode('ascii')
        .lower().strip()
    )


def _norm_col(col: str) -> str:
    """Normaliza nome de coluna: sem acento, minúsculo, espaços → underscore."""
    return re.sub(r'\s+', '_', _norm_txt(col))


def _find_first_col(df: pd.DataFrame, candidates_norm: set):
    """Retorna a primeira coluna do df cujo nome normalizado esteja em candidates_norm."""
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None


def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Remove colunas 'Unnamed:...' geradas pelo pandas ao ler CSVs com índice salvo."""
    cols_drop = [c for c in df.columns if _norm_col(c).startswith('unnamed')]
    return df.drop(columns=cols_drop).copy() if cols_drop else df.copy()


def _fmt_int(n) -> str:
    """Formata inteiro com separador de milhar brasileiro (ponto)."""
    try:
        return f'{int(n):,}'.replace(',', '.')
    except Exception:
        return str(n)


def _corrigir_grafia(texto: str) -> str:
    """Corrige grafia de categorias usando dicionário CORRECAO_GRAFIA."""
    chave = _norm_txt(texto)
    return CORRECAO_GRAFIA.get(chave, str(texto).strip())


def _normalizar_codigo_municipio(serie: pd.Series) -> pd.Series:
    """Normaliza código IBGE para 7 dígitos com zero à esquerda."""
    return (
        pd.to_numeric(serie, errors='coerce')
        .astype('Int64').astype(str)
        .str.replace('<NA>', '', regex=False)
        .str.zfill(7)
    )


def _centro_gdf(gdf) -> tuple:
    """Retorna (lat_centro, lon_centro) do bounding box de um GeoDataFrame."""
    b = gdf.total_bounds
    return (b[1] + b[3]) / 2, (b[0] + b[2]) / 2


# ── Helpers de escala dos gráficos ───────────────────────────────────────────

def _round_up_nice(v: float) -> int:
    """Arredonda v para cima para um número 'bonito' (múltiplo de step)."""
    v = float(v)
    if v <= 0:
        return 1000
    step = 200 if v <= 2000 else 500 if v <= 10000 else 1000
    return int(((v + step - 1) // step) * step)


def _build_ticks(xmax: int, n_ticks_max: int = 10) -> list:
    """Gera lista de ticks para o eixo X com no máximo n_ticks_max divisões."""
    raw_step  = xmax / n_ticks_max
    magnitude = 10 ** (len(str(int(raw_step))) - 1)
    step      = max(magnitude, round(raw_step / magnitude) * magnitude)
    return list(range(0, int(xmax) + step, step))


def _resolve_xmax(vals: list, fixo=None, folga: float = 0.45) -> int:
    """Calcula xmax do gráfico respeitando folga para rótulos."""
    d = max(vals) if vals else 0
    r = _round_up_nice(d * (1 + folga))
    return r if fixo is None else max(int(fixo), r)


# ── Categorias para mapas coropléticos ───────────────────────────────────────

def _categoria_casos(n: int, faixas: list) -> str:
    """Retorna o rótulo de faixa correspondente ao valor n."""
    if n == 0:
        return "Sem casos"
    prev = 0
    for f in faixas:
        if n <= f:
            return f"1–{f}" if prev == 0 else f"{prev + 1}–{f}"
        prev = f
    return f"> {faixas[-1]}"


def _build_ordem_categorias(faixas: list) -> list:
    """Gera lista ordenada de rótulos de categoria para a legenda do mapa."""
    return (
        ["Sem casos"]
        + [f"1–{faixas[0]}"]
        + [f"{faixas[i] + 1}–{faixas[i+1]}" for i in range(len(faixas) - 1)]
        + [f"> {faixas[-1]}"]
    )


def _build_cores_mapa(ordem: list) -> dict:
    """Associa cada categoria de ordem à cor da PALETA_MAPA."""
    cores = {"Sem casos": COR_SEM_CASOS}
    for label, cor in zip(reversed(ordem[1:]), PALETA_MAPA):
        cores[label] = cor
    return cores


print("✅ Funções auxiliares definidas.")

✅ Funções auxiliares definidas.


---
## 4. Carregamento e Filtragem dos Dados

Funções de leitura, filtragem e preparação das séries de casos para cada dimensão de análise.

In [4]:
def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica filtros de município, código IBGE e regional de saúde ao DataFrame."""
    out = _drop_index_cols(df)
    col_mun = _find_first_col(out, {'municipios','municipio','cidades','cidade'})
    col_cod = _find_first_col(out, {'codigo','cod'})
    col_reg = _find_first_col(out, {'regional_de_saude','regional_saude','regional'})
    if MUNICIPIO is not None and col_mun:
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]
    if CODIGO_MUNICIPIO is not None and col_cod:
        out = out[pd.to_numeric(out[col_cod], errors='coerce').astype('Int64') == int(CODIGO_MUNICIPIO)]
    if REGIONAL_SAUDE is not None and col_reg:
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]
    return out


def _year_cols(df: pd.DataFrame) -> list:
    """Retorna colunas cujo nome seja um ano válido (1900–2100)."""
    cols = []
    for c in df.columns:
        try:
            y = int(c)
            if 1900 <= y <= 2100:
                cols.append(c)
        except (ValueError, TypeError):
            pass
    return sorted(cols)


def normalizar_faixa_etaria(categoria: str) -> str:
    """Agrupa faixas acima de 70 anos em '70 ou mais'."""
    cat = _norm_txt(categoria)
    if cat in {'70-79','70 a 79','80+','80 ou mais','80 ou mais.','70 ou mais'}:
        return '70 ou mais'
    return str(categoria).strip()


def preparar_ano_diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega casos por ano a partir de colunas no formato YYYY."""
    ycols = _year_cols(df)
    s = df[ycols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0)
    s = s[s > 0]
    return (
        pd.DataFrame({'ano': s.index.astype(int), 'categoria': s.index.astype(str), 'total': s.values})
        .sort_values('ano').reset_index(drop=True)
    )


def preparar_long_wide(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega casos por (ano, categoria) a partir de colunas no formato YYYY_categoria."""
    rows = []
    for c in df.columns:
        if '_' not in str(c):
            continue
        ano_str, categoria = str(c).rsplit('_', 1)
        try:
            ano = int(ano_str)
        except (ValueError, TypeError):
            continue
        categoria = normalizar_faixa_etaria(categoria)
        total = pd.to_numeric(df[c], errors='coerce').fillna(0).sum()
        if total > 0:
            rows.append({'ano': ano, 'categoria': categoria, 'total': int(total)})
    return pd.DataFrame(rows).sort_values(['ano','categoria']).reset_index(drop=True)


def build_por_categoria_por_ano(df_long: pd.DataFrame, anos_range=None, top_n=None):
    """Agrega df_long em duas séries: por categoria (acumulado) e por ano."""
    d = df_long.copy()
    d['ano']   = pd.to_numeric(d['ano'],   errors='coerce').astype('Int64')
    d['total'] = pd.to_numeric(d['total'], errors='coerce')
    d = d.dropna(subset=['ano','categoria','total'])
    d['ano'] = d['ano'].astype(int)
    d = d[d['total'] > 0]
    if anos_range is not None:
        ai, af = map(int, anos_range)
        d = d[(d['ano'] >= ai) & (d['ano'] <= af)]
    por_categoria = d.groupby('categoria')['total'].sum()
    if top_n is not None:
        por_categoria = por_categoria.sort_values(ascending=False).head(int(top_n)).sort_values()
    por_ano = d.groupby('ano')['total'].sum().sort_index()
    return por_categoria, por_ano


def _carregar_e_preparar(caminho_csv: str, nome: str, modo: str = 'wide',
                          top_n=None, anos_range=ANOS_RANGE_CASOS):
    """Pipeline completo: lê CSV → filtra → prepara séries."""
    df   = aplicar_filtros(pd.read_csv(caminho_csv))
    print(f'  {nome}: {len(df)} linhas selecionadas')
    long = preparar_ano_diagnostico(df) if modo == 'ano_cols' else preparar_long_wide(df)
    return build_por_categoria_por_ano(long, anos_range=anos_range, top_n=top_n)


print("\n╔══════════════════════════════════════╗")
print("║       Configuração dos filtros       ║")
print("╠══════════════════════════════════════╣")
print(f"║  Município:        {str(MUNICIPIO):<20}║")
print(f"║  Código IBGE:      {str(CODIGO_MUNICIPIO):<20}║")
print(f"║  Regional:         {str(REGIONAL_SAUDE):<20}║")
print(f"║  Período:          {str(ANOS_RANGE_CASOS[0]) + '–' + str(ANOS_RANGE_CASOS[1]):<20}║")
print("╚══════════════════════════════════════╝")


╔══════════════════════════════════════╗
║       Configuração dos filtros       ║
╠══════════════════════════════════════╣
║  Município:        None                ║
║  Código IBGE:      None                ║
║  Regional:         None                ║
║  Período:          1985–2023           ║
╚══════════════════════════════════════╝


---
## 5. Análise Inicial dos Dados

Inspeção rápida de cada CSV de entrada para verificar dimensões, colunas e valores ausentes.

In [5]:
csvs_info = {
    'Ano Diagnóstico': CSV_CASOS_ANO_DIAGNOSTICO,
    'Escolaridade'   : CSV_CASOS_ESCOLARIDADE,
    'Raça/Cor'       : CSV_CASOS_RACA_COR,
    'Faixa Etária'   : CSV_CASOS_FAIXA_ETARIA,
}

for nome, path in csvs_info.items():
    df = pd.read_csv(path)
    missing = df.isnull().sum().sum()
    display(Markdown(f"**{nome}** — {df.shape[0]} linhas × {df.shape[1]} colunas | nulos totais: {missing}"))
    display(df.head(2))

**Ano Diagnóstico** — 853 linhas × 51 colunas | nulos totais: 1837

,MUNICÍPIOS,CÓDIGO,1985,1986,1987,1988,1989,1990,1991,1992,...,Sem Informacao,Unnamed: 19,Unnamed: 23,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 31,Unnamed: 32
0,abadia dos dourados,3100104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,abaete,3100203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


**Escolaridade** — 853 linhas × 163 colunas | nulos totais: 46454

,MUNICÍPIOS,CÓDIGO,1998_Sem Informacao,1999_Nivel medio,1999_Sem Informacao,2000_Fundamental completo,2000_Fundamental incompleto,2000_Nenhuma,2000_Nivel medio,2000_Nivel superior completo,...,2022_Nivel superior completo,2022_Nivel superior incompleto,2022_Sem Informacao,2023_Fundamental completo,2023_Fundamental incompleto,2023_Nenhuma,2023_Nivel medio,2023_Nivel superior completo,2023_Nivel superior incompleto,2023_Sem Informacao
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1,abaete,3100203,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,8.0,2.0,9.0,0.0,0.0,1.0,0.0,2.0


**Raça/Cor** — 853 linhas × 150 colunas | nulos totais: 45481

,MUNICÍPIOS,CÓDIGO,1998_Branca,1998_Parda,1998_Preta,1998_Sem Informacao,1999_Branca,1999_Preta,1999_Sem Informacao,2000_Amarela,...,2022_Indigena,2022_Parda,2022_Preta,2022_Sem Informacao,2023_Amarela,2023_Branca,2023_Indigena,2023_Parda,2023_Preta,2023_Sem Informacao
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,abaete,3100203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,0.0,11.0,0.0,1.0,0.0,2.0,0.0,11.0,1.0,0.0


**Faixa Etária** — 853 linhas × 306 colunas | nulos totais: 157159

,MUNICÍPIOS,CÓDIGO,1985_0-9,1985_10-19,1985_20-29,1985_30-39,1985_40-49,1985_50-59,1985_60-69,1985_70 ou mais,...,2022_60-69,2022_70 ou mais,2023_0-9,2023_10-19,2023_20-29,2023_30-39,2023_40-49,2023_50-59,2023_60-69,2023_70 ou mais
0,abadia dos dourados,3100104,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,abaete,3100203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,4.0,0.0,0.0,1.0,1.0,2.0,4.0,4.0,2.0


---
## 6. Funções de Visualização

Três funções de gráfico independentes com estética compartilhada:
- **`plot_ano_diagnostico`** — barras verticais: ano no eixo X, casos no eixo Y.
- **`plot_barras_categoria`** — barras horizontais: categoria no eixo Y, casos no eixo X.
- **`plot_dispersao_ano`** — dispersão: casos no eixo X, ano no eixo Y.

In [6]:
def _layout_base_casos(titulo: str, width: int, height: int) -> dict:
    """Retorna dict de layout compartilhado entre os gráficos de casos."""
    return dict(
        title=dict(
            text=titulo,
            font=dict(size=FONTE_TITULO, family="Arial", color="#1a1a2e"),
            x=0.5, xanchor="center",
        ),
        height=height, width=width,
        plot_bgcolor="#f2f4f7", paper_bgcolor="white",
        hovermode="closest",
        hoverlabel=dict(bgcolor="white", bordercolor="#aaa",
                        font=dict(size=FONTE_EIXO_TICK, family="Arial")),
    )


def plot_ano_diagnostico(
    por_ano: pd.Series,
    titulo: str = "Casos por Ano de Diagnóstico",
    ymax_fixo: int = 8000,
    width: int = LARGURA_GRAFICO,
    height: int = ALTURA_GRAFICO,
):
    """Barras verticais: ano no eixo X, número de casos no eixo Y."""
    s    = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    s    = s[s > 0].sort_index()
    anos = [str(a) for a in s.index.astype(int).tolist()]
    vals = s.values.astype(int).tolist()

    ymax  = max(ymax_fixo, int(max(vals) * 1.18))
    ticks = _build_ticks(ymax, n_ticks_max=8)

    fig = go.Figure(go.Bar(
        name="Total por ano",
        x=anos, y=vals,
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in vals],
        textposition="outside",
        textangle=0,
        textfont=dict(size=FONTE_TEXTO_BARRA, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
    ))

    layout = _layout_base_casos(titulo, width, height)
    layout["margin"] = dict(l=100, r=200, t=90, b=100)
    layout["xaxis"]  = dict(
        title="Ano do diagnóstico",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickangle=0,
        tickmode='array', tickvals=anos, ticktext=anos,
        showgrid=True, gridcolor='#d7dce3', gridwidth=1,
        automargin=True,
    )
    layout["yaxis"] = dict(
        title="Número de casos",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickmode='array',
        tickvals=ticks, ticktext=[_fmt_int(v) for v in ticks],
        range=[X_MIN, ymax],
        showgrid=True, gridcolor='#d7dce3', gridwidth=1,
        automargin=True,
    )
    fig.update_layout(**layout)
    fig.show()
    return fig


def plot_barras_categoria(
    por_categoria: pd.Series,
    titulo: str,
    ytitle: str,
    xmax_fixo: int,
    width: int = LARGURA_GRAFICO,
    height: int = ALTURA_GRAFICO,
):
    """Barras horizontais: categoria no eixo Y, número de casos no eixo X."""
    c = pd.to_numeric(por_categoria, errors='coerce').fillna(0)
    c = c[c > 0]

    # Garante ordem correta para faixa etária
    if _norm_txt(ytitle) == 'faixa etaria':
        ordem = [f for f in FAIXAS_ETARIAS_ORDEM if f in c.index.astype(str).tolist()]
        c = c.reindex(ordem)

    cats = [_corrigir_grafia(str(cat)) for cat in c.index.astype(str).tolist()]
    vals = c.values.astype(int).tolist()

    xmax_plot  = int(xmax_fixo)
    xmax_range = int(xmax_plot * (1 + FOLGA_ROTULOS))
    ticks      = _build_ticks(xmax_plot, n_ticks_max=10)

    fig = go.Figure(go.Bar(
        name="Total por categoria",
        x=vals, y=cats,
        orientation='h',
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in vals],
        textposition='outside',
        textfont=dict(size=FONTE_TEXTO_BARRA, color=COR_TEXTO_BARRA),
        cliponaxis=False,
        constraintext='none',
    ))

    layout = _layout_base_casos(titulo, width, height)
    layout["margin"] = dict(l=170, r=200, t=95, b=100)
    layout["xaxis"]  = dict(
        title="Número de casos",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        tickmode='array', tickvals=ticks, ticktext=[_fmt_int(v) for v in ticks],
        range=[X_MIN, xmax_range],
        showgrid=True, gridcolor='#d7dce3', gridwidth=1,
        automargin=True, tickangle=0,
    )
    layout["yaxis"] = dict(
        title=ytitle,
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=FONTE_EIXO_TICK),
        showgrid=True, gridcolor='#d7dce3', gridwidth=1,
        categoryorder='array', categoryarray=cats,
        automargin=True,
    )
    fig.update_layout(**layout)
    fig.show()
    return fig


def plot_dispersao_ano(
    por_ano: pd.Series,
    titulo: str,
    anos_range: tuple = ANOS_RANGE_CASOS,
    width: int = LARGURA_GRAFICO,
    height: int = 1050,
):
    """Dispersão: número de casos no eixo X, ano no eixo Y (todos os anos)."""
    s = pd.to_numeric(por_ano, errors='coerce').fillna(0)
    s = s[s > 0].sort_index()
    if s.empty:
        raise ValueError("Sem dados > 0 para plotar.")

    anos_lista = s.index.astype(int).tolist()
    valores    = s.values.astype(int).tolist()
    ano_ini, ano_fim = map(int, anos_range)

    x_max_val   = max(valores)
    x_max_plot  = int(x_max_val * 1.28)
    x_tick_step = 200 if x_max_val < 2000 else 500

    hover = [
        f"<b>Ano:</b> {ano}<br><b>Total de casos:</b> {_fmt_int(val)}"
        for ano, val in zip(anos_lista, valores)
    ]

    fig = go.Figure(go.Scatter(
        name=f"Total de casos por ano ({ano_ini}–{ano_fim})",
        x=valores, y=anos_lista,
        mode="markers+text",
        marker=dict(color=COR_PONTOS, size=9, opacity=0.90,
                    line=dict(color=COR_PONTOS, width=1), symbol="circle"),
        text=[_fmt_int(v) for v in valores],
        textposition="middle right",
        textfont=dict(size=12, color=COR_PONTOS),
        hovertext=hover, hoverinfo="text",
        hovertemplate="%{hovertext}<extra></extra>",
    ))

    xticks = list(range(0, x_max_plot + 1, x_tick_step))

    layout = _layout_base_casos(titulo, width, height)
    layout["margin"] = dict(l=160, r=120, t=100, b=80)
    layout["legend"] = dict(
        orientation="h", yanchor="bottom", y=1.02,
        xanchor="center", x=0.5,
        font=dict(size=FONTE_LEGENDA),
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#ddd", borderwidth=1,
    )
    layout["xaxis"] = dict(
        title="Número de casos",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickvals=xticks, ticktext=[_fmt_int(v) for v in xticks],
        tickfont=dict(size=FONTE_EIXO_TICK),
        gridcolor="#dde3ed", gridwidth=1,
        showline=True, linecolor="#ccc",
        zeroline=True, zerolinecolor="#bbb",
        range=[0, x_max_plot],
    )
    layout["yaxis"] = dict(
        title="Ano",
        title_font=dict(size=FONTE_EIXO_TITULO),
        tickfont=dict(size=12),
        showgrid=True, gridcolor="#dde3ed",
        showline=True, linecolor="#ccc",
        tickmode="array",
        tickvals=anos_lista,
        ticktext=[str(a) for a in anos_lista],
        range=[min(anos_lista) - 1, max(anos_lista) + 1],
        dtick=1,
    )
    fig.update_layout(**layout)
    fig.show()
    return fig


print("✅ Funções de visualização definidas.")

✅ Funções de visualização definidas.


---
## 7. Análise Exploratória Univariada

### 7.1 Casos por Ano de Diagnóstico

Série temporal do total anual de casos no período selecionado.

In [7]:
por_cat_ano, por_ano_diag = _carregar_e_preparar(
    CSV_CASOS_ANO_DIAGNOSTICO, 'Ano Diagnóstico',
    modo='ano_cols', top_n=None, anos_range=None,
)
por_ano_diag_filtrado = por_ano_diag[
    (por_ano_diag.index >= ANOS_RANGE_CASOS[0]) &
    (por_ano_diag.index <= ANOS_RANGE_CASOS[1])
]

plot_ano_diagnostico(
    por_ano=por_ano_diag_filtrado,
    titulo="Casos de Câncer de Mama por Ano de Diagnóstico",
    ymax_fixo=8000,
)

  Ano Diagnóstico: 853 linhas selecionadas


### 7.2 Casos por Faixa Etária

In [8]:
por_cat_faixa, por_ano_faixa = _carregar_e_preparar(
    CSV_CASOS_FAIXA_ETARIA, 'Faixa Etária',
    modo='wide', top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)

plot_barras_categoria(
    por_categoria=por_cat_faixa,
    titulo="Casos de Câncer de Mama por Faixa Etária",
    ytitle="Faixa etária",
    xmax_fixo=33000,
)

plot_dispersao_ano(
    por_ano=por_ano_faixa,
    titulo="Casos de Câncer de Mama por Ano — Faixa Etária",
)

  Faixa Etária: 853 linhas selecionadas


### 7.3 Casos por Escolaridade

In [9]:
por_cat_esc, por_ano_esc = _carregar_e_preparar(
    CSV_CASOS_ESCOLARIDADE, 'Escolaridade',
    modo='wide', top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)

plot_barras_categoria(
    por_categoria=por_cat_esc,
    titulo="Casos de Câncer de Mama por Escolaridade",
    ytitle="Escolaridade",
    xmax_fixo=37000,
)

plot_dispersao_ano(
    por_ano=por_ano_esc,
    titulo="Casos de Câncer de Mama por Ano — Escolaridade",
)

  Escolaridade: 853 linhas selecionadas


### 7.4 Casos por Raça/Cor

In [10]:
por_cat_raca, por_ano_raca = _carregar_e_preparar(
    CSV_CASOS_RACA_COR, 'Raça/Cor',
    modo='wide', top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)

plot_barras_categoria(
    por_categoria=por_cat_raca,
    titulo="Casos de Câncer de Mama por Raça/Cor",
    ytitle="Raça/Cor",
    xmax_fixo=55000,
)

plot_dispersao_ano(
    por_ano=por_ano_raca,
    titulo="Casos de Câncer de Mama por Ano — Raça/Cor",
)

  Raça/Cor: 853 linhas selecionadas


---
## 8. Tabelas de Distribuição

Distribuição percentual dos casos por cada dimensão categórica.

In [11]:
def _build_tabela(serie: pd.Series, col_categoria: str) -> pd.DataFrame:
    """Constrói tabela formatada com valor absoluto, percentual e linha de total."""
    total = int(serie.sum())
    df = serie.rename_axis(col_categoria).reset_index(name="Valor Absoluto")
    df["Valor Absoluto"] = df["Valor Absoluto"].astype(int)
    df["Percentual (%)"] = (df["Valor Absoluto"] / total * 100).map(
        lambda x: f"{x:,.2f}%".replace(",","X").replace(".","," ).replace("X",".")
    )
    df["Valor Absoluto"] = df["Valor Absoluto"].map(lambda x: f"{x:,}".replace(",", "."))
    total_row = pd.DataFrame([{
        col_categoria  : "Total de Casos",
        "Valor Absoluto": f"{total:,}".replace(",", "."),
        "Percentual (%)" : "100,00%",
    }])
    return pd.concat([df, total_row], ignore_index=True)


def _exibir_tabela(df: pd.DataFrame, caption: str) -> None:
    """Exibe tabela com estilo visual organizado."""
    n = len(df)
    styles = [
        {"selector": "caption", "props": [
            ("caption-side","top"),("font-weight","bold"),
            ("font-size","14px"),("text-align","center"),("padding-bottom","6px")]},
        {"selector": "th", "props": [
            ("border","1px solid #555"),("padding","5px 12px"),
            ("text-align","center"),("background-color","#e8eef6")]},
        {"selector": "td", "props": [
            ("border","1px solid #aaa"),("padding","4px 12px"),("text-align","center")]},
    ]
    styled = (
        df.style
        .set_caption(caption)
        .set_table_styles(styles)
        .apply(
            lambda row: ["font-weight: bold; border-top: 2px solid #333;"] * len(row)
            if row.name == n - 1 else [""] * len(row),
            axis=1,
        )
        .hide(axis="index")
    )
    display(styled)


# ── Faixa Etária ──────────────────────────────────────────────────────────────
c_faixa, _ = _carregar_e_preparar(
    CSV_CASOS_FAIXA_ETARIA, "Faixa Etária",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
c_faixa.index = [_corrigir_grafia(str(i)) for i in c_faixa.index]
c_faixa = pd.to_numeric(c_faixa, errors="coerce").fillna(0).astype(int)
c_faixa = c_faixa[c_faixa > 0].sort_values(ascending=False)
_exibir_tabela(_build_tabela(c_faixa, "Faixa Etária"),
               "Distribuição dos casos de câncer de mama por Faixa Etária em MG")

# ── Escolaridade ──────────────────────────────────────────────────────────────
c_esc, _ = _carregar_e_preparar(
    CSV_CASOS_ESCOLARIDADE, "Escolaridade",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
c_esc.index = [_corrigir_grafia(str(i)) for i in c_esc.index]
c_esc = pd.to_numeric(c_esc, errors="coerce").fillna(0).astype(int)
c_esc = c_esc[c_esc > 0].sort_values(ascending=False)
_exibir_tabela(_build_tabela(c_esc, "Escolaridade"),
               "Distribuição dos casos de câncer de mama por Escolaridade em MG")

# ── Raça/Cor ──────────────────────────────────────────────────────────────────
c_raca, _ = _carregar_e_preparar(
    CSV_CASOS_RACA_COR, "Raça/Cor",
    modo="wide", top_n=TOP_N_CATEGORIAS, anos_range=ANOS_RANGE_CASOS,
)
c_raca.index = [_corrigir_grafia(str(i)) for i in c_raca.index]
c_raca = pd.to_numeric(c_raca, errors="coerce").fillna(0).astype(int)
c_raca = c_raca[c_raca > 0].sort_values(ascending=False)
_exibir_tabela(_build_tabela(c_raca, "Raça/Cor"),
               "Distribuição dos casos de câncer de mama por Raça/Cor em MG")

# ── Ano de Diagnóstico ────────────────────────────────────────────────────────
df_ano_raw = aplicar_filtros(pd.read_csv(CSV_CASOS_ANO_DIAGNOSTICO))
ycols = [c for c in df_ano_raw.columns if str(c).isdigit()]
c_ano = df_ano_raw[ycols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=0)
c_ano.index = c_ano.index.astype(int)
c_ano = c_ano[c_ano > 0].sort_index(ascending=False)
c_ano.index = c_ano.index.astype(str)
_exibir_tabela(_build_tabela(c_ano, "Ano"),
               "Distribuição dos casos de câncer de mama por Ano de Diagnóstico em MG")

print(f"Total geral (Ano): {_fmt_int(int(c_ano.astype(int).sum()))}")

  Faixa Etária: 853 linhas selecionadas


Faixa Etária,Valor Absoluto,Percentual (%)
50-59,32.050,"26,69%"
40-49,27.156,"22,61%"
60-69,26.736,"22,26%"
70 ou mais,22.207,"18,49%"
30-39,10.428,"8,68%"
20-29,1.431,"1,19%"
10-19,55,"0,05%"
0-9,26,"0,02%"
Total de Casos,120.089,"100,00%"


  Escolaridade: 853 linhas selecionadas


Escolaridade,Valor Absoluto,Percentual (%)
Sem informação,36.639,"30,51%"
Fundamental incompleto,31.903,"26,57%"
Nível médio,18.782,"15,64%"
Fundamental completo,13.945,"11,61%"
Nível superior completo,12.138,"10,11%"
Nenhuma,5.527,"4,60%"
Nível superior incompleto,1.158,"0,96%"
Total de Casos,120.092,"100,00%"


  Raça/Cor: 853 linhas selecionadas


Raça/Cor,Valor Absoluto,Percentual (%)
Branca,53.683,"44,70%"
Parda,40.675,"33,87%"
Sem informação,16.411,"13,67%"
Preta,8.400,"6,99%"
Amarela,798,"0,66%"
Indígena,125,"0,10%"
Total de Casos,120.092,"100,00%"


Ano,Valor Absoluto,Percentual (%)
2023,5.923,"4,98%"
2022,7.088,"5,96%"
2021,6.985,"5,87%"
2020,6.107,"5,13%"
2019,6.991,"5,88%"
2018,6.951,"5,84%"
2017,6.466,"5,43%"
2016,6.652,"5,59%"
2015,5.882,"4,94%"
2014,5.988,"5,03%"


Total geral (Ano): 118.979


---
## 9. Mapas de Minas Gerais

Visualização espacial dos casos acumulados (1985–2023) em dois níveis de agregação:
- **Por município:** mapa coroplético colorido por classe de incidência.
- **Por regional de saúde:** dissolução dos municípios por regional.

A malha geográfica é obtida via `geobr` (IBGE) e integrada aos dados via código IBGE.

### 9.1 Localização dos arquivos auxiliares

In [12]:
candidatos_base = [Path.cwd().resolve()] + list(Path.cwd().resolve().parents)
path_municipios = None

for base in candidatos_base:
    pm = base / "Tratamento" / "Municípios" / "tabela_municipios_regionais.csv"
    if pm.exists():
        path_municipios = pm
        break

if path_municipios is None:
    raise FileNotFoundError(
        "Não foi possível localizar 'tabela_municipios_regionais.csv'. "
        "Verifique se existe em <raiz>/Tratamento/Municípios/."
    )

df_mun_reg = pd.read_csv(path_municipios)
df_mun_reg.columns              = [c.strip() for c in df_mun_reg.columns]
df_mun_reg["codigo_ibge"]       = pd.to_numeric(df_mun_reg["codigo_ibge"], errors="coerce").astype("Int64")
df_mun_reg["regional_original"] = df_mun_reg["regional_original"].fillna("Nenhum")

muni_mg = geobr.read_municipality(code_muni="MG", year=ANO_MALHA_MUNICIPIOS_MG, simplified=True).to_crs(4326)

print(f"✅ Malha geobr carregada: {len(muni_mg)} municípios")
print(f"✅ Tabela regionais carregada: {len(df_mun_reg)} registros")

✅ Malha geobr carregada: 853 municípios
✅ Tabela regionais carregada: 853 registros


### 9.2 Mapa por Município

In [13]:
df_casos_mapa = aplicar_filtros(pd.read_csv(CSV_CASOS_FAIXA_ETARIA))
df_casos_mapa.columns = [c.strip() for c in df_casos_mapa.columns]

anos_validos  = {str(a) for a in range(ANO_INI_MAPA, ANO_FIM_MAPA + 1)}
colunas_casos = [c for c in df_casos_mapa.columns if "_" in c and c.split("_")[0] in anos_validos]

col_cod_mapa = _find_first_col(df_casos_mapa, {"codigo","cod","código"}) or "CÓDIGO"
col_mun_mapa = _find_first_col(df_casos_mapa, {"municipios","municipio","municípios"}) or "MUNICÍPIOS"

df_casos_mapa["CODIGO_IBGE"] = pd.to_numeric(df_casos_mapa[col_cod_mapa], errors="coerce").astype("Int64")
df_casos_mapa["total_casos"] = (
    df_casos_mapa[colunas_casos]
    .apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1).astype(int)
)

ordem_mun  = _build_ordem_categorias(FAIXAS_MUNICIPIOS)
cores_mun  = _build_cores_mapa(ordem_mun)
df_casos_mapa["categoria_casos"] = df_casos_mapa["total_casos"].apply(
    lambda n: _categoria_casos(n, FAIXAS_MUNICIPIOS)
)

gdf_mun = muni_mg.merge(
    df_casos_mapa[["CODIGO_IBGE", col_mun_mapa, "total_casos", "categoria_casos"]],
    left_on="code_muni", right_on="CODIGO_IBGE", how="left",
)
gdf_mun["total_casos"]     = gdf_mun["total_casos"].fillna(0).astype(int)
gdf_mun["categoria_casos"] = pd.Categorical(
    gdf_mun["categoria_casos"].fillna("Sem casos"),
    categories=ordem_mun, ordered=True,
)
gdf_mun = gdf_mun.reset_index(drop=True)
gdf_mun["id"] = gdf_mun.index.astype(str)

lat_c, lon_c = _centro_gdf(gdf_mun)

fig_mun = px.choropleth_map(
    gdf_mun,
    geojson=json.loads(gdf_mun.to_json()),
    locations="id",
    color="categoria_casos",
    color_discrete_map=cores_mun,
    category_orders={"categoria_casos": ordem_mun},
    title=f"Casos Acumulados de Câncer de Mama por Município em MG ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="name_muni",
    hover_data={"total_casos": True, "categoria_casos": False, "id": False},
    labels={"total_casos": "Total de casos"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5, opacity=0.95,
)
fig_mun.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5, height=760,
    legend_title_text="Nº de casos",
)
fig_mun.show()
print("✅ Mapa por município gerado.")

✅ Mapa por município gerado.


### 9.3 Mapa por Regional de Saúde

In [ ]:
# Associação município → regional via código IBGE
base_reg = df_casos_mapa[["CODIGO_IBGE","total_casos"]].merge(
    df_mun_reg[["codigo_ibge","regional_original"]],
    left_on="CODIGO_IBGE", right_on="codigo_ibge", how="left",
)
base_reg["regional_original"] = base_reg["regional_original"].fillna("Nenhum")

por_regional = (
    base_reg[base_reg["regional_original"] != "Nenhum"]
    .groupby("regional_original", as_index=False)
    .agg(total_casos=("total_casos","sum"))
)
por_regional["total_casos"] = por_regional["total_casos"].astype(int)

ordem_reg  = _build_ordem_categorias(FAIXAS_REGIONAIS)
cores_reg  = _build_cores_mapa(ordem_reg)
por_regional["categoria_casos"] = por_regional["total_casos"].apply(
    lambda n: _categoria_casos(n, FAIXAS_REGIONAIS)
)

# Dissolve municípios por regional com CRS projetado para distância correta
gdf_mg_reg = muni_mg.merge(
    df_mun_reg[["codigo_ibge","regional_original"]],
    left_on="code_muni", right_on="codigo_ibge", how="left",
)
gdf_mg_reg["regional_original"] = gdf_mg_reg["regional_original"].fillna("Nenhum")

# Preenchimento geoespacial para municípios sem regional (EPSG:31983 = SIRGAS2000/UTM23S)
sem_reg_mask = gdf_mg_reg["regional_original"] == "Nenhum"
if sem_reg_mask.any():
    gdf_proj     = gdf_mg_reg.to_crs(epsg=31983)
    com_regional = gdf_proj[~sem_reg_mask].copy()
    for idx in gdf_mg_reg[sem_reg_mask].index:
        geom = gdf_proj.loc[idx, "geometry"]
        dist = com_regional.geometry.distance(geom)
        gdf_mg_reg.loc[idx, "regional_original"] = com_regional.loc[dist.idxmin(), "regional_original"]

gdf_regionais = (
    gdf_mg_reg[gdf_mg_reg["regional_original"] != "Nenhum"][["regional_original","geometry"]]
    .dissolve(by="regional_original", as_index=False)
)
gdf_regionais = gdf_regionais.merge(por_regional, on="regional_original", how="left")
gdf_regionais["total_casos"]     = gdf_regionais["total_casos"].fillna(0).astype(int)
gdf_regionais["categoria_casos"] = pd.Categorical(
    gdf_regionais["categoria_casos"].fillna("Sem casos"),
    categories=ordem_reg, ordered=True,
)
gdf_regionais = gdf_regionais.reset_index(drop=True)
gdf_regionais["id"] = gdf_regionais.index.astype(str)

lat_c, lon_c = _centro_gdf(gdf_regionais)

fig_reg = px.choropleth_map(
    gdf_regionais,
    geojson=json.loads(gdf_regionais.to_json()),
    locations="id",
    color="categoria_casos",
    color_discrete_map=cores_reg,
    category_orders={"categoria_casos": ordem_reg},
    title=f"Casos Acumulados de Câncer de Mama por Regional de Saúde em MG ({ANO_INI_MAPA}–{ANO_FIM_MAPA})",
    hover_name="regional_original",
    hover_data={"total_casos": True, "categoria_casos": False, "id": False},
    labels={"total_casos": "Total de casos"},
    map_style="white-bg",
    center={"lat": lat_c, "lon": lon_c},
    zoom=5.5, opacity=0.95,
)
fig_reg.update_layout(
    margin=dict(l=10, r=10, t=60, b=10),
    title_x=0.5, height=760,
    legend_title_text="Nº de casos",
)
fig_reg.show()
print("✅ Mapa por regional gerado.")

---
## 10. Conclusões da EDA — Casos

### Distribuição por faixa etária
- A incidência concentra-se nas faixas acima de 40 anos, com pico entre 50–69 anos, condizente com a literatura epidemiológica.

### Distribuição por escolaridade
- Categorias com menor escolaridade e "Sem informação" aparecem com frequência elevada, refletindo limitações do registro nos sistemas públicos.

### Distribuição por raça/cor
- A categoria "Parda" e "Branca" concentram a maior parte dos registros, acompanhando a composição populacional de Minas Gerais.

### Tendência temporal
- A série por Ano de Diagnóstico evidencia crescimento consistente ao longo do período, com possível efeito de aumento do diagnóstico e rastreamento.

### Distribuição espacial
- A região metropolitana de BH e o Triângulo Mineiro concentram os maiores totais absolutos de casos.

> **Próxima etapa (PFC II):** Aplicação de K-means clustering e análise bivariada entre incidência e variáveis socioeconômicas (IDH, renda, acesso a serviços de saúde).